## **TAHAP 1** *Membangun Case Base*

# Set Up dan Download Library

In [ ]:
!pip install pymupdf pandas scikit-learn nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 36.9 MB/s eta 0:00:00


In [ ]:
import os

os.makedirs("/content/data/pdf", exist_ok=True)
os.makedirs("/content/data/raw", exist_ok=True)
os.makedirs("/content/data/processed/clean_text", exist_ok=True)
os.makedirs("/content/data/eval", exist_ok=True)
os.makedirs("/content/data/logs", exist_ok=True)

print("SETUP DONE")

SETUP DONE


# PDF ke TXT

In [ ]:
import fitz
import os

input_dir = "/content/data/pdf"
output_dir = "/content/data/raw"

count = 0
failed = []

for file in os.listdir(input_dir):

    if file.endswith(".pdf"):

        try:
            doc = fitz.open(os.path.join(input_dir, file))
            text = ""

            for page in doc:
                text += page.get_text()

            doc.close()

            out_path = os.path.join(output_dir, file.replace(".pdf", ".txt"))

            with open(out_path, "w", encoding="utf-8") as f:
                f.write(text)

            count += 1

        except:
            failed.append(file)

print("BERHASIL:", count)
print("GAGAL:", len(failed))

BERHASIL: 34
GAGAL: 0


# Cek hasil Raw

In [ ]:
with open("/content/data/raw/case_001.txt", encoding="utf-8") as f:
    print(f.read()[:1000])

hkama
ahkamah Agung Repub
ahkamah Agung Republik Indonesia
mah Agung Republik Indonesia
blik Indonesi
Direktori Putusan Mahkamah Agung Republik Indonesia
putusan.mahkamahagung.go.id
P U T U S A N
Nomor 747/Pid.B/2025/PN Mtr
DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA
Pengadilan  Negeri  Mataram yang  mengadili  perkara  pidana  dengan
acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai
berikut dalam perkara Terdakwa :
1. Nama lengkap 
: ABDULLAH
2. Tempat lahir 
: Mentigi
3. Umur/Tanggal lahir 
: 46 Tahun /31 Desember 1979
4. Jenis kelamin 
: Laki-laki
5. Kebangsaan 
: Indonesia
6. Tempat tinggal 
: Dusun Mentigi, Desa Malaka, Kecamatan 
Pemenang, Kabupaten Lombok Utara
7. Agama 
: Islam
8. Pekerjaan 
: Wiraswasta
Terdakwa Abdullah ditahan dalam rumah tahanan oleh : 
1. Penyidik, sejak tanggal 29 Juli 2025 sampai dengan tanggal 17 Agustus
2025. 
2. Penyidik, Perpanjangan Oleh Penuntut Umum sejak tanggal 18 Agustus 2025
sampai dengan tanggal 26 September 2025

# Cleaning Text

In [ ]:
import re
import os

def clean_text(text):

    text = text.lower()

    # FIX PDF GARBAGE
    text = re.sub(r'\ba\s*h\s*agung\s*repub.*?indonesia\b',
                  'mahkamah agung republik indonesia', text)

    text = re.sub(r'direktori putusan.*?go\.id', ' ', text)
    text = re.sub(r'putusan\.mahkamahagung.*?go\.id', ' ', text)

    # hapus header umum
    text = re.sub(r'hkama', ' ', text)

    # hapus karakter aneh
    text = re.sub(r'[^a-z0-9\s\.\,\;\:\-\(\)]', ' ', text)

    # normalisasi spasi
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


raw_dir = "/content/data/raw"
clean_dir = "/content/data/processed/clean_text"

os.makedirs(clean_dir, exist_ok=True)

for file in os.listdir(raw_dir):
    if file.endswith(".txt"):

        with open(os.path.join(raw_dir, file), "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        cleaned = clean_text(text)

        with open(os.path.join(clean_dir, file), "w", encoding="utf-8") as f:
            f.write(cleaned)

print("CLEANING FIXED DONE")

CLEANING FIXED DONE


# Hasil Cleaning

In [ ]:
file_path = "/content/data/processed/clean_text/case_001.txt"

with open(file_path, encoding="utf-8") as f:
    print(f.read()[:1000])

a h agung repub a h agung republik indonesia mah agung republik indonesia blik indonesi direktori putusan ma h agung republik indonesia p u t u s a n nomor 747 pid.b 2025 pn mtr demi keadilan berdasarkan ketuhanan yang maha esa pengadilan negeri mataram yang mengadili perkara pidana dengan acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai berikut dalam perkara terdakwa : 1. nama lengkap : abdullah 2. tempat lahir : mentigi 3. umur tanggal lahir : 46 tahun 31 desember 1979 4. jenis kelamin : laki-laki 5. kebangsaan : indonesia 6. tempat tinggal : dusun mentigi, desa malaka, kecamatan pemenang, kabupaten lombok utara 7. agama : islam 8. pekerjaan : wiraswasta terdakwa abdullah ditahan dalam rumah tahanan oleh : 1. penyidik, sejak tanggal 29 juli 2025 sampai dengan tanggal 17 agustus 2025. 2. penyidik, perpanjangan oleh penuntut umum sejak tanggal 18 agustus 2025 sampai dengan tanggal 26 september 2025. 3. penuntut umum, sejak tanggal 26 september 2025 sampai denga

In [ ]:
print("RAW FILES:", len(os.listdir("/content/data/raw")))
print("CLEAN FILES:", len(os.listdir("/content/data/processed/clean_text")))

RAW FILES: 34
CLEAN FILES: 34


# VALIDASI KEUTUHAN TEKS


In [ ]:
valid = []
invalid = []

for file in os.listdir(clean_dir):

    if not file.endswith(".txt"):
        continue

    path = os.path.join(clean_dir, file)

    with open(path, encoding="utf-8") as f:
        text = f.read()

    if len(text.split()) > 500:
        valid.append(file)
    else:
        invalid.append(file)

print("VALID:", len(valid))
print("INVALID:", len(invalid))

if len(valid) >= 30:
    print("MEMENUHI MINIMAL 30 DOKUMEN")
else:
    print("BELUM MEMENUHI MINIMAL 30 DOKUMEN")

VALID: 34
INVALID: 0
MEMENUHI MINIMAL 30 DOKUMEN
